In [95]:
import numpy as np
import pandas as pd
import pwlf
# === 参数 ===
excel_file = '../data/test_data.xlsx'  # Excel 文件名（与脚本放同目录）
fit_start_date = '2025-04-04'
fit_end_date = '2025-04-08'
new_date= '2025-04-09'
n_segments = 3  # CoPiLinear 拟合段数
# === 读取数据 ===
df = pd.read_excel(excel_file)

In [96]:
# === 拟合模型 ===
fit_df = df[(df['date'] >= fit_start_date) & (df['date'] <= fit_end_date)]
x = fit_df['load_rate'].values
y = fit_df['price'].values

# 定义权重：同样对 x<=0.3 和 x>0.7 数据赋予较大权重
weights = np.ones_like(x)
weights[x <= 0.3] = 3.0
weights[x > 0.68] = 3.0

# 初始参数猜测
model = pwlf.PiecewiseLinFit(x, y, weights=weights)
# 拟合3段线性函数（不需要在 fit 中再传 weights 参数）
breaks = model.fit(n_segments)  

In [97]:
# === 数据清洗 ===
processing_df = df[df['date'] == new_date].copy()
x_processing = processing_df['load_rate'].values
y_processing = processing_df['price'].values
y_pred = model.predict(x_processing)
# === 异常值检测（基于残差和 IQR） ===
residuals = y_processing - y_pred
Q1 = np.percentile(residuals, 25)
Q3 = np.percentile(residuals, 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

processing_df['predicted_price'] = y_pred
processing_df['residual'] = residuals
processing_df['is_outlier'] = (residuals < lower_bound) | (residuals > upper_bound)

processing_df[processing_df['is_outlier']]

,date,time_slot,price,load_rate,predicted_price,residual,is_outlier
845,2025-04-09,19:30,395.0,92.8,827.685615,-432.685615,True
846,2025-04-09,19:45,395.0,92.9,848.890735,-453.890735,True
